In [ ]:
# 인스타 크롤링
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import undetected_chromedriver as uc
import time , tempfile 
import sys   
import math   
import pandas as pd   
import os
import random
import urllib.request
import urllib
from openpyxl.styles import Font
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
import chromedriver_autoinstaller

print('='*80)
print('< 이 프로그램은 인스타그램에서 애니 성지순례 데이터를 수집하는 프로그램 입니다.>')
print('='*80)

qu = "https://www.instagram.com/accounts/login/"

now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(now.tm_year, now.tm_mon, now.tm_mday, \
                                      now.tm_hour, now.tm_min, now.tm_sec)

ff = 'c:\\py_temp\\2차프로젝트\\'
if ff == '':
    ff = 'c:\\py_temp\\2차프로젝트\\'

img_dir = ff + s + '-' + '인스타그램_애니성지순례' + '\\img'

ft_name = ff+s+'-'+'인스타그램_애니성지순례'+'\\'+s+'-'+'인스타그램_애니성지순례'+'.txt'
fx_name = ff+s+'-'+'인스타그램_애니성지순례'+'\\'+s+'-'+'인스타그램_애니성지순례'+'.xlsx'
fc_name = ff+s+'-'+'인스타그램_애니성지순례'+'\\'+s+'-'+'인스타그램_애니성지순례'+'.csv'

email = input('id를 입력하세요: ')
if email == '':
    email = 'kmsoo1224@naver.com'
    
pw = input('비밀번호를 입력하세요: ')

search1 = input('검색어: ')

if search1 == '':
    search1 = '#애니성지순례'

In [18]:
chromedriver_autoinstaller.install(path='c:/py_temp')
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

driver.maximize_window()

# 🔥 인스타는 qu(검색결과/프로필/게시물)로 바로 가지 말고,
# 로그인 화면이 떠도 안정적으로 잡히게 "로그인 페이지"로 먼저 유도
driver.get("https://www.instagram.com/accounts/login/")
driver.switch_to.default_content()

# input 뜰 때까지 대기 (name이 안 먹히는 날 대비: autocomplete도 같이)
user_sel = (By.CSS_SELECTOR, "input[name='username'], input[autocomplete='username']")
pass_sel = (By.CSS_SELECTOR, "input[name='password'], input[autocomplete='current-password']")

iid2 = wait.until(EC.presence_of_element_located(user_sel))
ipwd2 = wait.until(EC.presence_of_element_located(pass_sel))

In [19]:
iid2.clear()
for a in email:
    iid2.send_keys(a)
    time.sleep(0.3)

ipwd2.clear()
for b in pw:
    ipwd2.send_keys(b)
    time.sleep(0.3)

# 로그인 버튼은 type=submit이 제일 안정적
wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']"))).click()

In [20]:
# iid2 = driver.find_element(By.NAME,'username')
# for a in email:
#     iid2.send_keys(a)
#     time.sleep(0.5)
# ipwd2 = driver.find_element(By.NAME,'password')
# for b in pw:
#     ipwd2.send_keys(b)
#     time.sleep(0.5)

# #로그인 클릭
# wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="loginForm"]/div/div[3]/button/div'))).click()
# time.sleep(2)

In [ ]:
#돋보기 클릭
wait.until(EC.element_to_be_clickable((By.LINK_TEXT,'검색'))).click()
time.sleep(2)

selector = (
    "input[type='text'][placeholder='검색'],"
    "input[type='text'][aria-label*='검색'],"
    "[role='textbox'][aria-label*='검색']"
)
search_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))

for c in search1:
    search_box.send_keys(c)
    time.sleep(0.5)

tag_text = search1 if search1.startswith("#") else f"#{search1}"

el = wait.until(EC.element_to_be_clickable(
    (By.XPATH, f'//a[.//*[contains(normalize-space(), "{tag_text}")]]')
))
driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
driver.execute_script("arguments[0].click();", el)

tag = search1.lstrip("#")
driver.get(f"https://www.instagram.com/explore/tags/{tag}/")

In [ ]:
import os, time
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By

# =========================
# 0) 게시글 링크 수집 함수
# =========================
def collect_post_links(driver, limit=200, scroll_times=20, sleep=1.5):
    links = []
    seen = set()

    for _ in range(scroll_times):
        a_tags = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/p/"]')
        for a in a_tags:
            href = a.get_attribute("href")
            if href and "/p/" in href and href not in seen:
                seen.add(href)
                links.append(href)

        if len(links) >= limit:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep)

    return links[:limit]

# =========================
# 1) (너 기존) 초기 세팅
# =========================
no = 1

no2 = []
id2 = []
write2 = []
url2 = []

os.makedirs(img_dir, exist_ok=True)
os.chdir(img_dir)

# =========================
# 2) 태그 페이지에서 링크 수집
#    (현재 driver가 태그 페이지에 들어와 있는 상태여야 함)
# =========================
post_links = collect_post_links(driver, limit=300, scroll_times=40, sleep=2)
print("수집된 게시글 링크 수:", len(post_links))

# =========================
# 3) 링크 하나씩 들어가서 추출 (너 스타일 유지)
# =========================
for r, url in enumerate(post_links, start=1):
    driver.get(url)
    time.sleep(2)  # 렌더링 대기

    html = driver.page_source
    s = BeautifulSoup(html, 'html.parser')
    c1 = s.find('div', 'x6s0dn4 x78zum5 xdt5ytf xdj266r x11t971q xat24cr xvc5jky x1n2onr6 xh8yej3')

    # c1이 못 잡히면 스킵
    if c1 is None:
        print(f"[{r}/{len(post_links)}] c1 못찾음 -> 스킵: {url}")
        continue

    no2.append(no)
    print('1. 번호: ' + str(no))

    # 아이디
    try:
        id3 = c1.find('span', 'xt0psk2').get_text()
    except:
        id3 = ""
    id2.append(id3)
    print('2. 아이디: ' + id3)

    # 본문내용 + 해시태그
    try:
        write1 = c1.find('span', 'x193iq5w xeuugli x13faqbe x1vvkbs xt0psk2 x1i0vuye xvs91rp xo1l8bm x5n08af x10wh9bi xpm28yp x8viiok x1o7cslx x126k92a').get_text()
    except:
        write1 = ""
    write2.append(write1)
    print('3. 내용: ' + write1 + '\n' + '='*90 + '\n')

    url2.append(url)
    print('4. 링크: ' + url)

    time.sleep(1)
    no += 1

print("완료! 추출된 개수:", len(no2))

In [ ]:
# 1) DF 만들기 (url2가 있다고 가정)
sunggi = pd.DataFrame()
sunggi['번호'] = no2
sunggi['아이디'] = pd.Series(id2)
sunggi['내용'] = pd.Series(write2)
sunggi['게시글 링크'] = pd.Series(url2)   # ✅ 일단 URL 문자열로 저장

# 2) 엑셀로 저장
sunggi.to_excel(fx_name, index=False)

# 3) openpyxl로 다시 열어서 "하이퍼링크"로 변환
wb = load_workbook(fx_name)
ws = wb.active

link_col = sunggi.columns.get_loc("게시글 링크") + 1  # openpyxl은 1부터 시작

for row in range(2, ws.max_row + 1):  # 1행은 헤더
    cell = ws.cell(row=row, column=link_col)
    url = cell.value
    if isinstance(url, str) and url.startswith("http"):
        cell.hyperlink = url              # ✅ 하이퍼링크 객체 지정
        cell.style = "Hyperlink"          # ✅ 엑셀 기본 하이퍼링크 스타일(파란색/밑줄)
        # 또는 수동 폰트 적용하고 싶으면:
        # cell.font = Font(color="0000FF", underline="single")

wb.save(fx_name)

# CSV 저장은 그대로(CSV는 하이퍼링크 “객체” 개념이 없음)
sunggi.to_csv(fc_name, encoding="utf-8-sig", index=False)

print('='*80)
print('저장을 완료하였습니다. (엑셀 하이퍼링크 적용 완료)')
print('='*80)